In [27]:
import pandas as pd  # Pandas is our data manipulation tool (like Excel for Python)
import numpy as np   # Numpy helps us do complex math fast

# Scikit-Learn (sklearn) is the main library we use for Machine Learning
from sklearn.model_selection import train_test_split # Used to split our data into "study" and "test" piles
from sklearn.pipeline import Pipeline                # Helps us chain together different steps into one smooth workflow
from sklearn.compose import ColumnTransformer        # Lets us apply specific changes to specific columns
from sklearn.preprocessing import StandardScaler     # Shrinks down large numbers (like income) so they don't overpower small numbers (like years)
from sklearn.impute import SimpleImputer             # Automatically fills in any blank/missing data cells
from sklearn.linear_model import LogisticRegression  # The actual math algorithm making the Yes/No decisions
from sklearn.metrics import accuracy_score           # The grading rubric to tell us how well our model did

print("Cell 1 complete: Libraries imported.")

Cell 1 complete: Libraries imported.


In [28]:
# Step 1: Read the data. 'df' stands for DataFrame, which is essentially a table of data.
df = pd.read_csv("dataset.csv")

# We define what clues (features) the model gets to look at to make a decision
feature_cols = ["income", "expenses", "loanamount", "term_years"]

# The "target" is the answer we want the model to guess.
# Datasets name this differently, so we check for a few common names.
possible_targets = ["can_get_loan", "loan_approved", "approved", "target", "label"]

# Sometimes the dataset has slightly different column names.
# This standardizes the "term" column so our code doesn't break.
if "term_years" not in df.columns:
    for alt in ["term", "term_of_loan", "loan_term", "tenure_years"]:
        if alt in df.columns:
            df = df.rename(columns={alt: "term_years"})
            break

# Search our DataFrame to find exactly which target column exists in this dataset
target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    raise ValueError(
        f"Couldn't find target column. Tried: {possible_targets}\n"
        f"Columns: {list(df.columns)}"
    )

# Double-check that we aren't missing any required clues (features)
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required feature columns: {missing}")

# In machine learning, 'X' usually represents the clues (features)
# and 'y' usually represents the answer key (target).
X = df[feature_cols].copy()
y = df[target_col].copy()

# Machine learning models only understand math (numbers).
# If the answer key (y) says "Yes" or "No", we must convert it to 1s and 0s.
if not np.issubdtype(y.dtype, np.number):
    # factorize() automatically turns text categories into numbers
    y, mapping = pd.factorize(y)
    print(f"Target mapping used: {dict(enumerate(mapping))}")

print("Cell 2 complete: Dataset loaded and features/target separated.")

Cell 2 complete: Dataset loaded and features/target separated.


In [29]:
# Imagine giving a student a practice test and the real exam.
# If you give them the real exam to study, they'll just memorize the answers!
# So, we take 80% of our data for "studying" (X_train, y_train)
# and hide 20% for the "final exam" (X_test, y_test).

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,    # 20% goes to the final exam
    random_state=42,  # A random seed so we get the exact same split every time we run it
    stratify=y        # Ensures the 20% has a fair mix of Approved and Rejected loans
)

print(f"Cell 3 complete: Data split. Training size: {len(X_train)}, Testing size: {len(X_test)}")


Cell 3 complete: Data split. Training size: 40000, Testing size: 10000


In [30]:
# A pipeline is a set of instructions that the data goes through in order.

# Step A: How to handle the numbers (Preprocessing)
numeric_pipeline = Pipeline(steps=[
    # Imputer: If someone left a box blank on their loan form, fill it with the median (middle) value
    ("imputer", SimpleImputer(strategy="median")),
    # Scaler: An income of $100,000 looks way bigger to a computer than a term of 15 years.
    # Scaler shrinks everything down to a similar scale so the computer treats them fairly.
    ("scaler", StandardScaler()),
])

# Step B: Apply our numeric rules to our specific feature columns
preprocess = ColumnTransformer(
    transformers=[("num", numeric_pipeline, feature_cols)]
)

# Step C: Put the preprocessor and the actual AI Brain (Logistic Regression) together
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,           # Give the model up to 2000 tries to figure out the best math
        class_weight="balanced", # If the dataset has way more rejections than approvals, this forces the model to care equally about both
        C=1.0,                   # Regularization: Prevents the model from memorizing the data too closely (prevents over-studying)
        solver="lbfgs"           # The specific math formula used to solve the problem
    )),
])

print("Cell 4 complete: Machine learning pipeline is ready.")

Cell 4 complete: Machine learning pipeline is ready.


In [31]:
print("Training the model...")

# The .fit() command is the most important command in machine learning.
# This is where the model looks at the training clues (X_train)
# and the correct answers (y_train) and learns the patterns.
clf.fit(X_train, y_train)

print("Cell 5 complete: Training finished!")

Training the model...
Cell 5 complete: Training finished!


In [32]:
# Let's see how well the student learned!
# We ask it to guess the answers for the data it studied, AND the data we hid.

train_pred = clf.predict(X_train) # Guesses on the study guide
test_pred = clf.predict(X_test)   # Guesses on the final exam

# Compare the guesses to the actual real-world answers to get a grade (accuracy)
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

# The gap tells us if the model memorized the study guide (overfitting)
# If it scores 99% on training but 60% on testing, it just memorized the training data!
gap = train_acc - test_acc

print(f"Training Accuracy: {train_acc:.4f} (Score on studied data)")
print(f"Testing Accuracy : {test_acc:.4f} (Score on hidden test data)")
print(f"Train-Test Gap   : {gap:.4f}")

if gap > 0.05:
    print("\nWarning: Possible overfitting (training accuracy is much higher than testing).")
else:
    print("\nModel looks healthy. No major overfitting detected.")

print("\nCell 6 complete.")


Training Accuracy: 0.8936 (Score on studied data)
Testing Accuracy : 0.8868 (Score on hidden test data)
Train-Test Gap   : 0.0068

Model looks healthy. No major overfitting detected.

Cell 6 complete.


In [ ]:
print("\n--- Interactive Loan Predictor ---")
print("Enter values as YEARLY amounts. Type Ctrl+C to stop.\n")

while True:
    try:
        # 1. Ask the human for their financial details
        income = float(input("Yearly income: "))
        expenses = float(input("Yearly expenses: "))
        loanamount = float(input("Loan amount: "))
        term_years = float(input("Loan term (years): "))

        # 2. Package those details into a DataFrame (the exact same format the AI studied)
        user_df = pd.DataFrame([{
            "income": income,
            "expenses": expenses,
            "loanamount": loanamount,
            "term_years": term_years
        }])

        # 3. Ask the AI for its final prediction (0 or 1)
        pred = int(clf.predict(user_df)[0])

        # 4. Ask the AI how confident it is (Percentage chance of approval)
        # predict_proba returns [Probability of 0, Probability of 1]. We want the second one.
        prob_approve = float(clf.predict_proba(user_df)[0][1])

        # Print the results out nicely for the user
        print(f"\n>> Prediction: {'APPROVED' if pred == 1 else 'REJECTED'}")
        print(f">> Approval Probability: {prob_approve:.2%}\n")

        # Ask if they want to try another person
        again = input("Check another loan? (y/n): ").strip().lower()
        if again != "y":
            print("Exiting predictor.")
            break
        print("-" * 30)

    except KeyboardInterrupt:
        # Handles what happens if the user presses "Stop" in Colab
        print("\nExiting predictor.")
        break
    except ValueError:
        # Handles what happens if the user types letters instead of numbers
        print("\n[!] Error: Please enter valid numbers only.\n")


--- Interactive Loan Predictor ---
Enter values as YEARLY amounts. Type Ctrl+C to stop.

Yearly income: 100000
Yearly expenses: 45000
Loan amount: 450000
Loan term (years): 15

>> Prediction: REJECTED
>> Approval Probability: 0.00%

